# 车辆颜色识别 — 数据探索与实验记录

本 Notebook 用于：
1. 了解数据集分布
2. 可视化预处理效果（CLAHE + 双边滤波）
3. 观察 HSV 特征分布
4. 直观展示模型性能

In [ ]:
import sys
sys.path.insert(0, '../src')

import cv2
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

from preprocess import preprocess, apply_clahe, apply_bilateral_filter
from feature_extraction import extract_hsv_histogram, H_BINS, S_BINS, V_BINS
from utils import load_dataset, print_class_distribution, LABEL_MAP, ID_TO_ZH

print('模块加载成功 ✅')

## 1. 数据集分布统计

In [ ]:
images, labels = load_dataset(verbose=True)
print_class_distribution(labels)

In [ ]:
from collections import Counter

counter = Counter(labels)
names = [ID_TO_ZH[i] for i in sorted(counter.keys())]
counts = [counter[i] for i in sorted(counter.keys())]

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.bar(names, counts, color=['#2c3e50','#27ae60','#f39c12','#e74c3c','#bdc3c7'])
ax.set_title('各颜色类别样本数量分布', fontsize=14)
ax.set_ylabel('样本数量')
for bar, count in zip(bars, counts):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
            str(count), ha='center', fontsize=10)
plt.tight_layout()
plt.show()

## 2. 预处理效果可视化

对比原始图像 vs CLAHE增强 vs 双边滤波后的效果

In [ ]:
def show_preprocess_comparison(bgr_img):
    """对比预处理各步骤效果"""
    raw_resized = cv2.resize(bgr_img, (256, 256))
    after_clahe = apply_clahe(raw_resized)
    after_bilateral = apply_bilateral_filter(after_clahe)
    final = preprocess(bgr_img)

    titles = ['原始图像', 'CLAHE 增强', '双边滤波', '最终结果']
    imgs = [raw_resized, after_clahe, after_bilateral, final]

    fig, axes = plt.subplots(1, 4, figsize=(14, 4))
    for ax, img, title in zip(axes, imgs, titles):
        ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
        ax.set_title(title)
        ax.axis('off')
    plt.suptitle('预处理流水线各步骤对比', fontsize=13)
    plt.tight_layout()
    plt.show()

# 随机选一张图展示
if images:
    show_preprocess_comparison(images[0])

## 3. HSV 特征直方图可视化

对比不同颜色车辆的 H/S/V 通道分布，直观展示为什么 HSV 比 RGB 更适合颜色识别

In [ ]:
def plot_hsv_hist(bgr_img, title=''):
    """可视化单张图的 HSV 通道直方图"""
    processed = preprocess(bgr_img)
    feat = extract_hsv_histogram(processed)

    h_feat = feat[:H_BINS]
    s_feat = feat[H_BINS:H_BINS+S_BINS]
    v_feat = feat[H_BINS+S_BINS:]

    fig, axes = plt.subplots(1, 3, figsize=(12, 3))
    axes[0].bar(range(H_BINS), h_feat, color='cornflowerblue')
    axes[0].set_title('H（色调）分布')
    axes[1].bar(range(S_BINS), s_feat, color='coral')
    axes[1].set_title('S（饱和度）分布')
    axes[2].bar(range(V_BINS), v_feat, color='gold')
    axes[2].set_title('V（亮度）分布')
    plt.suptitle(title)
    plt.tight_layout()
    plt.show()

# 每个类别展示一张
from collections import defaultdict
by_class = defaultdict(list)
for img, lbl in zip(images, labels):
    by_class[lbl].append(img)

for lbl in sorted(by_class.keys()):
    plot_hsv_hist(by_class[lbl][0], title=f'{ID_TO_ZH[lbl]} 颜色车辆 — HSV 直方图')

## 4. 训练并展示混淆矩阵

> 如果已经运行过 `python src/train.py`，直接加载结果图即可

In [ ]:
from pathlib import Path

cm_path = Path('../results/confusion_matrix.png')
fi_path = Path('../results/feature_importance.png')

if cm_path.exists():
    img = plt.imread(str(cm_path))
    plt.figure(figsize=(8, 6))
    plt.imshow(img)
    plt.axis('off')
    plt.title('混淆矩阵')
    plt.show()
else:
    print('请先运行: python src/train.py')

if fi_path.exists():
    img = plt.imread(str(fi_path))
    plt.figure(figsize=(10, 6))
    plt.imshow(img)
    plt.axis('off')
    plt.title('特征重要性')
    plt.show()